# 15. 불용어 적용 전후 HS 코드 분류 성능 비교

03번 실험의 평가 설계를 그대로 사용해 TF-IDF + Linear SVM의 불용어 적용 전후를 비교합니다. 입력은 제품명·국문 원재료·영문 원재료·물품 형태 및 포장·용도 5개 속성만 사용합니다. 판정요지 원문은 정답 코드 누출을 막기 위해 입력에서 제외합니다.

불용어는 Kiwi 형태소 분석으로 국문 속성에서 완전 일치하는 형태소만 제거합니다. 영문 원재료와 나머지 텍스트는 유지합니다. 동일한 StratifiedGroupKFold 분할과 모델 설정을 사용합니다.


In [ ]:
!pip -q install pandas numpy scikit-learn kiwipiepy


In [ ]:
import pandas as pd, numpy as np, re, json
from urllib.parse import quote
from kiwipiepy import Kiwi
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
DATA_FILE = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv', safe='')
df = pd.read_csv(BASE + DATA_FILE, encoding='utf-8-sig').fillna('')
INPUT_COLUMNS = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET_COLUMN = 'HS Code'
KOREAN_COLUMNS = ['Product Name','Main Ingredients (KR)','Product Form and Packaging','Intended Use']
for col in INPUT_COLUMNS + [TARGET_COLUMN]: df[col] = df[col].astype(str).str.strip()
df = df[df[TARGET_COLUMN].ne('')].reset_index(drop=True)
df['model_text_raw'] = '[PRODUCT] ' + df['Product Name'] + ' [INGREDIENTS_KR] ' + df['Main Ingredients (KR)'] + ' [INGREDIENTS_EN] ' + df['Main Ingredients (EN)'] + ' [FORM] ' + df['Product Form and Packaging'] + ' [USE] ' + df['Intended Use']
df['duplicate_group'] = pd.util.hash_pandas_object(df[INPUT_COLUMNS], index=False).astype(str)
print('전체 사례:',len(df),'| HS 코드 수:',df[TARGET_COLUMN].nunique())


## 1. 평가 데이터 및 동일 Fold 구성

03번과 동일하게 사례 수 4건 이상, 고유 입력 그룹 4개 이상인 코드를 주 평가에 포함합니다. 4-fold StratifiedGroupKFold와 random_state=42를 고정합니다.


In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()
eligible_labels = class_counts[class_counts >= 4].index
eval_df = df[df[TARGET_COLUMN].isin(eligible_labels)].copy()
group_counts = eval_df.groupby(TARGET_COLUMN)['duplicate_group'].nunique()
eligible_labels = group_counts[group_counts >= 4].index
eval_df = eval_df[eval_df[TARGET_COLUMN].isin(eligible_labels)].reset_index(drop=True)
X_raw = eval_df['model_text_raw'].reset_index(drop=True)
y = eval_df[TARGET_COLUMN].reset_index(drop=True)
groups = eval_df['duplicate_group'].reset_index(drop=True)
labels = np.sort(y.unique())
splits = list(StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42).split(X_raw,y,groups))
fold_audit = []
for fold,(tr,te) in enumerate(splits,1): fold_audit.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':len(set(groups.iloc[tr]) & set(groups.iloc[te])),'test_classes_missing_from_train':len(set(y.iloc[te])-set(y.iloc[tr]))})
fold_audit_df = pd.DataFrame(fold_audit)
display(fold_audit_df)
print('평가 사례:',len(eval_df),'| 평가 코드:',len(labels))
if (fold_audit_df['group_overlap'] != 0).any(): raise ValueError('Train/test 그룹 중복 발생')


## 2. 불용어 전처리

14번 실험에서 정리한 22개 후보를 적용합니다. 형태소의 표면형이 불용어와 완전히 일치할 때만 제거하며, 단어 내부 문자열은 삭제하지 않습니다. 원문은 보존합니다.


In [ ]:
stopwords = {'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
kiwi = Kiwi()
def remove_stopword_morphs(text):
    text = str(text)
    tokens = kiwi.tokenize(text)
    chars = list(text)
    for token in reversed(tokens):
        if token.form in stopwords:
            for i in range(token.start, token.start + token.len): chars[i] = ' '
    return ''.join(chars)
eval_df['model_text_stopword'] = eval_df['model_text_raw']
for col in KOREAN_COLUMNS:
    eval_df[col + '_clean'] = eval_df[col].map(remove_stopword_morphs)
eval_df['model_text_stopword'] = '[PRODUCT] ' + eval_df['Product Name_clean'] + ' [INGREDIENTS_KR] ' + eval_df['Main Ingredients (KR)_clean'] + ' [INGREDIENTS_EN] ' + eval_df['Main Ingredients (EN)'] + ' [FORM] ' + eval_df['Product Form and Packaging_clean'] + ' [USE] ' + eval_df['Intended Use_clean']
X_stop = eval_df['model_text_stopword'].reset_index(drop=True)
display(pd.DataFrame({'raw':X_raw.head(5),'stopword_applied':X_stop.head(5)}))
print('불용어 후보 수:',len(stopwords))


## 3. TF-IDF + Linear SVM 비교

03번의 설정을 유지합니다: 단어 n-gram (1,2), 문자 n-gram (2,5), 각 최대 특징 수 30,000, sublinear_tf=True, LinearSVC(C=1.0, class_weight='balanced'). Fold마다 벡터라이저를 새로 적합합니다.


In [ ]:
def make_vectorizer():
    return FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))])
predictions = []
fold_metrics = []
for fold,(train_idx,test_idx) in enumerate(splits,1):
    y_train,y_test = y.iloc[train_idx],y.iloc[test_idx]
    for model_name,X in [('Baseline',X_raw),('Stopword',X_stop)]:
        vectorizer = make_vectorizer()
        x_train = vectorizer.fit_transform(X.iloc[train_idx])
        x_test = vectorizer.transform(X.iloc[test_idx])
        model = LinearSVC(C=1.0,class_weight='balanced').fit(x_train,y_train)
        pred = model.predict(x_test)
        scores = model.decision_function(x_test)
        classes = model.classes_
        top3 = [[classes[i] for i in np.argsort(row)[::-1][:min(3,len(classes))]] for row in scores]
        fold_metrics.append({'model':model_name,'fold':fold,'accuracy':accuracy_score(y_test,pred),'macro_f1':f1_score(y_test,pred,labels=labels,average='macro',zero_division=0),'weighted_f1':f1_score(y_test,pred,average='weighted',zero_division=0),'top3_accuracy':np.mean([true in ranked for true,ranked in zip(y_test,top3)])})
        for idx,true,guess,ranked in zip(test_idx,y_test,pred,top3): predictions.append({'model':model_name,'row_index':int(idx),'fold':fold,'true_hs':true,'predicted_hs':guess,'correct':bool(true==guess),'top3_candidates':ranked})
fold_metrics_df = pd.DataFrame(fold_metrics)
predictions_df = pd.DataFrame(predictions)
display(fold_metrics_df)


## 4. 종합 성능 및 차이

전체 OOF 예측을 합쳐 Accuracy, Macro-F1, Weighted-F1, Top-3 Accuracy를 계산합니다. 동일한 테스트 사례에 대한 두 모델의 예측을 직접 비교합니다.


In [ ]:
summary_rows = []
for name,part in predictions_df.groupby('model'):
    summary_rows.append({'model':name,'accuracy':accuracy_score(part['true_hs'],part['predicted_hs']),'macro_f1':f1_score(part['true_hs'],part['predicted_hs'],labels=labels,average='macro',zero_division=0),'weighted_f1':f1_score(part['true_hs'],part['predicted_hs'],average='weighted',zero_division=0),'top3_accuracy':np.mean([true in ranked for true,ranked in zip(part['true_hs'],part['top3_candidates'])])})
summary_df = pd.DataFrame(summary_rows).set_index('model')
display(summary_df)
delta = (summary_df.loc['Stopword']-summary_df.loc['Baseline']).rename('Stopword - Baseline').to_frame()
display(delta)
wide = predictions_df.pivot(index='row_index',columns='model',values='predicted_hs').reset_index()
wide = wide.merge(eval_df[['HS Code']],left_on='row_index',right_index=True)
wide['baseline_correct'] = wide['Baseline'].eq(wide['HS Code'])
wide['stopword_correct'] = wide['Stopword'].eq(wide['HS Code'])
print('예측 변경:',int((wide['Baseline']!=wide['Stopword']).sum()))
print('오류→정답:',int((~wide['baseline_correct'] & wide['stopword_correct']).sum()))
print('정답→오류:',int((wide['baseline_correct'] & ~wide['stopword_correct']).sum()))
display(wide[(wide['Baseline']!=wide['Stopword'])].head(50))


## 5. HS 코드별 성능

Macro-F1 변화가 특정 HS 코드에 의해 좌우되는지 확인합니다. 클래스별 Support와 F1을 함께 비교합니다.


In [ ]:
class_rows = []
for name,part in predictions_df.groupby('model'):
    report = classification_report(part['true_hs'],part['predicted_hs'],labels=labels,output_dict=True,zero_division=0)
    for code in labels:
        class_rows.append({'model':name,'HS Code':code,'support':int(report.get(code,{}).get('support',0)),'precision':report.get(code,{}).get('precision',0),'recall':report.get(code,{}).get('recall',0),'f1':report.get(code,{}).get('f1-score',0)})
class_metrics_df = pd.DataFrame(class_rows)
class_compare = class_metrics_df.pivot(index='HS Code',columns='model',values='f1').fillna(0)
class_compare['delta'] = class_compare['Stopword']-class_compare['Baseline']
display(class_compare.sort_values('delta'))


## 6. 결과 저장

Fold별 성능, 종합 성능, 사례별 OOF 예측, HS 코드별 성능, Fold 누수 점검 결과를 저장합니다.


In [ ]:
OUTPUT_DIR = Path('/content/hs_stopword_comparison')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
fold_metrics_df.to_csv(OUTPUT_DIR/'15_fold_metrics.csv',index=False,encoding='utf-8-sig')
summary_df.to_csv(OUTPUT_DIR/'15_summary_metrics.csv',encoding='utf-8-sig')
predictions_df.to_csv(OUTPUT_DIR/'15_oof_predictions.csv',index=False,encoding='utf-8-sig')
class_metrics_df.to_csv(OUTPUT_DIR/'15_class_metrics.csv',index=False,encoding='utf-8-sig')
wide.to_csv(OUTPUT_DIR/'15_prediction_changes.csv',index=False,encoding='utf-8-sig')
fold_audit_df.to_csv(OUTPUT_DIR/'15_fold_audit.csv',index=False,encoding='utf-8-sig')
with open(OUTPUT_DIR/'15_stopword_config.json','w',encoding='utf-8') as f: json.dump({'stopwords':sorted(stopwords)},f,ensure_ascii=False,indent=2)
print('저장 완료:',OUTPUT_DIR)
from google.colab import files
files.download(str(OUTPUT_DIR/'15_summary_metrics.csv'))


## 7. 해석 기준

- Macro-F1을 주요 비교 지표로 보고 Accuracy·Weighted-F1·Top-3도 함께 확인합니다.
- Fold 평균과 표준편차를 함께 확인해 특정 Fold에 따른 변동을 살핍니다.
- 성능이 좋아져도 144건 내외의 소규모 평가이므로 일반화 성능 향상으로 단정하지 않습니다.
- 성능이 하락하면 전체 불용어를 일괄 확정하지 말고 `분류·제품·해당·호·소호` 등 핵심 후보를 묶음별로 ablation하는 후속 실험을 진행합니다.

**주의:** 이 비교는 제품 속성 5개를 입력으로 하는 기존 HS 코드 분류 모델의 전처리 민감도를 평가합니다. 판정요지 문장 자체를 입력으로 넣는 실험이 아닙니다.
